## Part 6 — Q9: text/image causal coupling (no-code runner)

**For Q9 only, start here. Skip all earlier cells, including the older setup and Drive mount.**
Select an A100 runtime and run just **1. Setup** and **2. Run Q9 pilot**. The second cell configures, runs and displays figures. Skip optional advanced/export cells. Do not use whole-notebook Run all in the main notebook, which would also launch the older experiments.

- **smoke**: first-run wiring/audit check, not scientific evidence.
- **discovery**: clean T5/DiT measurements and calibration; no edited generations.
- **screen**: paired single-forward interventions from clean latent states, not final edited images.
- **confirm**: complete edited images. Pilot has 12 edited trajectories; full has 576 and requires the long-run checkbox.

**Default pilot:** three evaluation prompts, one seed; three separate calibration prompts, one seed; block 17 at step 0; four text-state comparisons. Screening runs 6 clean trajectories plus 12 edited forwards (and 3 replay checks). Pilot confirm runs at most 18 full trajectories including calibration. No separate smoke/discovery run is required; calibration is automatic. Skipping smoke does not mean a pretrained GPU run has been verified.

No Python edits or external config file are required. Checkpoint, scheduler steps, guidance, prompts, seeds and intervention settings come from the model/mode presets. Optional form fields let you narrow the experiment after screening. Presets are hypotheses, not automatic discovery of the correct sites.

FLUX.1-dev and Schnell are supported. PixArt has no evolving DiT text read-back pathway.
For gated FLUX.1-dev, accept its Hugging Face license and add a Colab secret named **HF_TOKEN** (or use the login widget).
Raw activations/replay states remain in memory, full images stay in temporary `/content`, and Drive export is **off by default**, limited to 250 MiB of compact artifacts per run. Runtime reset loses local work.

[Experimental design and interpretation limits](https://github.com/BrendanGho/massive-activations-fig3/blob/main/SPEC_Q9.md)

The pilot disables extra routing, donor, random-direction, channel, reverse, rescue, empty-prompt and layer/time-sweep conditions. It tests only pre-birth text influence; it cannot establish maintenance, read-back, prompt-general stability or independence. Three calibration prompts are exploratory, not a robust direction fit. Full experiments remain available via the workload dropdown. Resolution, the model's denoising step count and pairing controls are unchanged.


### What are we actually testing?

**Main question:** if we change a candidate text state, does the image-register circuit change downstream?
The intervention menu is an operationalization of our Q9 plan, not a claim that every method comes from the original image-only paper or is already validated by it. Extra controls distinguish several explanations for the same observed effect.

| Experiment | Plain-language question |
| --- | --- |
| Remove the shared text direction (`remove_direction`) | Does this particular component of the text state matter for the image circuit? |
| Match the norm change (`norm_matched`) | Is the effect simply because the edited text state got smaller? This preserves direction while matching the norm after direction removal; it is not Q7's ordinary-token-median scaling. |
| Zero candidate states (`zero`) | Do these candidate text positions matter at all? This is a broad perturbation, not a selective semantic deletion. |
| Zero matched ordinary states (`ordinary_zero`) | Are candidates more important than comparable noncandidate text positions? Unavailable matches are reported, not invented. |
| Block attention pathways | Is the effect carried from text to image, or from image back to text? Score and value variants distinguish changing attention allocation from removing a contribution. |
| Perturb image registers (`image_*`) | Does changing the image circuit affect later text states? |
| Restore image-register states (rescue) | After a text perturbation, can restoring image registers recover later measurements or the image? Recovery only at the patched site is not evidence. |

Channel suppression, random-direction controls and donor swaps are additional checks of what component/content matters. You do not need to interpret every method at once. The default pilot runs only the first four comparisons. The full smoke/screen presets retain the broad menu, and full confirm adds rescue variants.

### What do the modes mean?

| Mode | What runs | What it can tell you |
| --- | --- | --- |
| `smoke` — check wiring | A small test of capture, edits, replay and audits | Whether the implementation runs; not a scientific conclusion |
| `discovery` — observe | Clean-model calibration and text/image measurements; no interventions | Candidate states, directions, channels and routing to investigate |
| `screen` — probe cheaply | Replay one transformer forward from a saved clean diffusion state, apply an edit, and measure its downstream effects | Within-forward causal effects; not the effect on a completed edited image |
| `confirm` — generate images | Full same-seed trajectories with a selected step/layer intervention and optional rescue | Whether internal effects persist through generation and affect the image |

These are **workflow stages**, not early/middle/late diffusion phases. Screen still needs full clean baseline trajectories; it saves work on the many edited trials. Discovery does not automatically pick sites, and screen does not automatically select the confirmation experiments. Review results, then choose the confirmation fields explicitly; unchanged fields use the predefined hypothesis-based preset.

For this pilot, start with **screen** (calibration runs automatically). If a contrast is useful, use pilot **confirm** on three new held-out prompts. Expand to full discovery/routing/read-back tests later. Small effects or no effects in this restricted pilot do not settle the full Q9 question.


In [ ]:
# @title 1. Q9 setup and authentication (self-contained; run once)
import os, subprocess, sys
from pathlib import Path
Q9_REPO_DIR = '/content/massive-activations-fig3'
if not Path(Q9_REPO_DIR).exists():
    subprocess.run(['git', 'clone', '--branch', 'main',
                    'https://github.com/BrendanGho/massive-activations-fig3.git', Q9_REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', Q9_REPO_DIR, 'pull', '--ff-only', 'origin', 'main'], check=True)
os.chdir(Q9_REPO_DIR)
if Q9_REPO_DIR not in sys.path:
    sys.path.insert(0, Q9_REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[q9]'], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Choose Runtime > Change runtime type > GPU, then rerun Q9 setup.')
print(torch.cuda.get_device_name(), 'VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1))
if torch.cuda.memory_allocated() > 2**30:
    print('Other experiments retain GPU memory. For best results, restart the session and run only Part 6.')
from huggingface_hub import login
from google.colab import userdata
try:
    q9_hf_token = userdata.get('HF_TOKEN')
except Exception:
    q9_hf_token = None
if q9_hf_token:
    login(token=q9_hf_token, add_to_git_credential=False)
else:
    login(add_to_git_credential=False)
print('Q9 setup ready. No Drive mount is needed.')

q9_advanced = {}
q9_finished = False
print('Next run 2. Run Q9 pilot. Setup resets any previous advanced overrides.')


### Optional field guide

Use commas for indices/seeds/methods (for example `17`, `0,14,27`, or `zero,ordinary_zero`).
Use `||` between custom prompts. Calibration and evaluation prompts must not overlap.
Prompt count `0` keeps all preset/custom prompts; a positive number takes the first N. Reducing counts makes a smaller pilot, not full confirmation.

Text methods: `remove_direction,norm_matched,suppress_channel,zero,ordinary_zero,random_direction,donor_swap`.
Routing: `image_reads_text_score,image_reads_text_value,text_reads_register_score,text_reads_register_value,text_reads_content_score,text_reads_content_value`.
Reverse: `image_remove_direction,image_zero,image_ordinary_zero`.
Rescues require confirm mode, text-state methods and a rescue layer downstream of every site. Select rescues `none` for routing/reverse confirmation. Readout must follow every site.
LPIPS/CLIP apply to confirm images; structured scores require an external evaluator CSV and are optional.
If you do not need these controls, **skip the guide and advanced cell**.


In [ ]:
# @title Optional advanced form (run BEFORE step 2 only if needed) — leave blank/0/preset to keep defaults
# Sites are zero-based blocks (-1 = projected T5); steps are zero-based denoising indices.
Q9_SITES = '' # @param {type:"string"}
Q9_STEPS = '' # @param {type:"string"}
Q9_METHODS = '' # @param {type:"string"}
Q9_PROMPT_COUNT = 0 # @param {type:"integer"}
Q9_SEEDS = '' # @param {type:"string"}
Q9_CUSTOM_PROMPTS = '' # @param {type:"string"}
Q9_CALIBRATION_PROMPT_COUNT = 0 # @param {type:"integer"}
Q9_CALIBRATION_SEEDS = '' # @param {type:"string"}
Q9_CUSTOM_CALIBRATION_PROMPTS = '' # @param {type:"string"}
Q9_CANDIDATE_CLASSES = 'eos,pad' # @param ['eos,pad', 'content', 'content,eos,pad,special']
Q9_CANDIDATE_SOURCE = 'union' # @param ['norm', 'sink', 'union', 'intersection']
Q9_RESCUES = 'preset' # @param ['preset', 'none', 'none,projection,state,ordinary_projection,sham']
Q9_RESCUE_LAYER = '' # @param {type:"string"}
Q9_READOUT_LAYER = '' # @param {type:"string"}
Q9_ATTENTION_LAYERS = '' # @param {type:"string"}
Q9_INCLUDE_EMPTY = 'preset' # @param ['preset', 'yes', 'no']
Q9_MEMORY = 'auto' # @param ['auto', 'offload', 'gpu']
Q9_OPTIMIZE_PROBES = True # @param {type:"boolean"}
Q9_SKIP_UNAVAILABLE = True # @param {type:"boolean"}
Q9_EVALUATE_LPIPS = True # @param {type:"boolean"}
Q9_EVALUATE_CLIP = True # @param {type:"boolean"}
Q9_STRUCTURED_SCORES = '' # @param {type:"string"}
q9_advanced = dict(
    sites=Q9_SITES, steps=Q9_STEPS, methods=Q9_METHODS, prompt_count=Q9_PROMPT_COUNT,
    seeds=Q9_SEEDS, prompts=Q9_CUSTOM_PROMPTS, calibration_prompt_count=Q9_CALIBRATION_PROMPT_COUNT,
    calibration_seeds=Q9_CALIBRATION_SEEDS, calibration_prompts=Q9_CUSTOM_CALIBRATION_PROMPTS,
    candidate_classes=Q9_CANDIDATE_CLASSES, candidate_source=Q9_CANDIDATE_SOURCE,
    rescues=Q9_RESCUES, rescue_layer=Q9_RESCUE_LAYER, readout_layer=Q9_READOUT_LAYER,
    attention_layers=Q9_ATTENTION_LAYERS, include_empty=Q9_INCLUDE_EMPTY, memory=Q9_MEMORY,
    optimize_probes=Q9_OPTIMIZE_PROBES, skip_unavailable=Q9_SKIP_UNAVAILABLE,
    evaluate_lpips=Q9_EVALUATE_LPIPS, evaluate_clip=Q9_EVALUATE_CLIP,
    structured_scores=Q9_STRUCTURED_SCORES)
print('Advanced overrides recorded. Step 2 validates them before loading a model.')


In [ ]:
# @title 2. Run Q9 pilot — config, execution and figures in one cell
Q9_MODEL = 'flux1-dev' # @param ['flux1-dev', 'flux-schnell']
Q9_MODE = 'screen' # @param ['screen', 'discovery', 'confirm', 'smoke']
Q9_WORKLOAD = 'pilot' # @param ['pilot', 'full']
Q9_RESOLUTION = 'preset' # @param ['preset', '512', '1024']
Q9_RUN_EXPERIMENT = True # @param {type:"boolean"}
Q9_CONFIRM_FULL_RUN = False # @param {type:"boolean"}
# The long-run checkbox applies only to full confirmation, not the 12-job pilot.
import json
from dataclasses import asdict
from src.experiments.q9_colab import build_config, run_budget, show_results
if 'Q9_REPO_DIR' not in globals():
    raise RuntimeError('Run Q9 step 1 (setup) first.')
q9_finished = False
q9_result = None
cfg = build_config(Q9_MODEL, Q9_MODE, Q9_RESOLUTION, workload=Q9_WORKLOAD,
                   vram_gib=torch.cuda.get_device_properties(0).total_memory / 2**30,
                   bf16=torch.cuda.is_bf16_supported(), advanced=globals().get('q9_advanced', {}))
budget = run_budget(cfg)
print('Workload:', Q9_WORKLOAD, '(advanced overrides apply last)')
print(json.dumps(budget, indent=2))
print('Upper bounds before resume/unavailable skips; donor replays may add forwards.')
print('Sites:', cfg.sites, '| steps:', cfg.steps, '| methods:', cfg.methods, '| rescues:', cfg.rescues)
print('Evaluation prompts:', cfg.prompts, '| seeds:', cfg.seeds)
print('Dtype:', cfg.dtype, '| offload:', cfg.offload, '| local outputs:', cfg.output_dir)
if globals().get('q9_advanced'):
    print('Advanced overrides:', q9_advanced)
Q9_CONFIG_PATH = f'/content/q9_{cfg.model_preset}_{cfg.mode}_config.json'
Path(Q9_CONFIG_PATH).write_text(json.dumps(asdict(cfg), indent=2), encoding='utf-8')
if not Q9_RUN_EXPERIMENT:
    print('Preview only. Enable Q9_RUN_EXPERIMENT to execute.')
elif cfg.mode == 'confirm' and budget['edited_full_trajectories'] > 12 and not Q9_CONFIRM_FULL_RUN:
    print('Not started. This exceeds the 12-job pilot; review the budget and enable Q9_CONFIRM_FULL_RUN.')
else:
    subprocess.run([sys.executable, '-u', '-m', 'src.experiments.text_image_coupling',
                    '--config', Q9_CONFIG_PATH], check=True)
    q9_finished = True
    q9_result = show_results(cfg)
    print('Pilot results are exploratory; no-candidate/unavailable controls are not negative evidence.')


In [ ]:
# @title Optional: redisplay figures (already shown by step 2)
from IPython.display import display, Image
if not globals().get('q9_finished', False):
    print('No completed run in this workflow yet. Run step 2 first.')
else:
    from src.experiments.q9_report import locate
    q9_result = locate(cfg)
    print((q9_result / 'report_status.json').read_text())
    for figure in sorted((q9_result / 'figures').glob('*.png')):
        print(figure.name)
        display(Image(filename=str(figure)))
    print('Results:', q9_result)
    print('Check audits.csv and direction_stability.csv before interpreting a negative result.')
    print('No-candidate/unavailable controls are excluded. Smoke is a wiring check only.')


In [ ]:
# @title Optional compact Drive export (no raw activations or full image grid)
Q9_EXPORT_TO_DRIVE = False # @param {type:"boolean"}
Q9_DRIVE_ROOT = '/content/drive/MyDrive/Research/MA/q9_compact' # @param {type:"string"}
if not Q9_EXPORT_TO_DRIVE:
    print('Drive export off. Results remain in temporary Colab storage.')
elif not globals().get('q9_finished', False):
    print('Run step 2 successfully before exporting.')
else:
    from google.colab import drive
    drive.mount('/content/drive')
    subprocess.run([sys.executable, '-m', 'src.experiments.text_image_coupling',
                    '--config', Q9_CONFIG_PATH, '--export-compact', Q9_DRIVE_ROOT], check=True)
